# GraphRAG vs Vector RAG for organizational knowledge

This notebook models the knowledge of **Halcyon Systems**, a fictional software company, as a graph. The graph holds people, skills, tools, projects, decisions and documents. The notebook then answers the same questions two ways:

- **Vector RAG** embeds document chunks and retrieves the top-k chunks by cosine similarity.
- **GraphRAG** links the question to graph nodes, walks typed relationships out 2 hops, and answers from the facts and linked documents it finds.

The goal is to show *when structured relationships matter more than semantic similarity*.

Sections 1, 2 and 4 run offline. The other sections call Claude, so they need `ANTHROPIC_API_KEY` set.

In [ ]:
import genacademy_p2_rag_pipeline  # sets up the OS cert store for HTTPS
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

from genacademy_p2_rag_pipeline import llm
from genacademy_p2_rag_pipeline.data import load_kb, load_questions
from genacademy_p2_rag_pipeline.graph import build_graph, stats, neighbors, k_hop_triples, triples_to_text
from genacademy_p2_rag_pipeline.graph_rag import GraphRAG
from genacademy_p2_rag_pipeline.vector_rag import VectorRAG

kb = load_kb()
questions = {q.id: q for q in load_questions()}
g = build_graph(kb)
node_counts, edge_counts = stats(g)
print(f"{g.number_of_nodes()} nodes, {g.number_of_edges()} edges")
pd.DataFrame({"nodes": pd.Series(node_counts)}).T

## 1. Graph schema
The edge types are the relationships that vector search can only infer from prose.

In [ ]:
pd.Series(edge_counts).sort_values(ascending=False).to_frame("count").T

## 2. The whole graph

In [ ]:
KIND_COLORS = {"person": "#2a78d6", "project": "#e5484d", "decision": "#e8a33d",
               "tool": "#3aa76d", "skill": "#8e6fd8", "document": "#8a8f98"}

def draw(graph, highlight=(), title="", ax=None, seed=7):
    ax = ax or plt.subplots(figsize=(14, 10))[1]
    pos = nx.spring_layout(graph, k=0.9, seed=seed)
    colors = [KIND_COLORS[graph.nodes[n]["kind"]] for n in graph]
    sizes = [700 if n in highlight else 300 for n in graph]
    nx.draw_networkx_edges(graph, pos, ax=ax, alpha=0.25, arrows=True, arrowsize=8)
    nx.draw_networkx_nodes(graph, pos, ax=ax, node_color=colors, node_size=sizes,
                           edgecolors=["black" if n in highlight else "white" for n in graph])
    nx.draw_networkx_labels(graph, pos, {n: graph.nodes[n]["name"][:18] for n in graph}, ax=ax, font_size=7)
    for kind, c in KIND_COLORS.items():
        ax.scatter([], [], color=c, label=kind)
    ax.legend(loc="lower left", frameon=False)
    ax.set_title(title); ax.axis("off")

draw(g, title="Halcyon Systems knowledge graph")

## 3. Why vector search struggles with structure

**q07** asks which people have both Python and LLM evaluation skills. No document states this. The answer exists only as `HAS_SKILL` edges, so vector retrieval can't find it, while the graph answers it with a set intersection.

In [ ]:
vector, graph_rag = VectorRAG(kb), GraphRAG(kb)

q = questions["q07"]
print("Vector top chunks:", [(c.doc_id, round(s, 3)) for c, s in vector.retrieve(q.question)])
both = neighbors(g, "s01", {"HAS_SKILL"}) & neighbors(g, "s12", {"HAS_SKILL"})
print("Graph (set intersection):", [kb.name(p) for p in both])

In [ ]:
def compare(qid):
    q = questions[qid]
    v, gr = vector.answer(q.question), graph_rag.answer(q.question)
    print(f"[{qid} | {q.type}] {q.question}")
    print(f"Expected: {q.expected_answer}")
    print()
    print(f"VECTOR: {v['answer']}")
    print()
    print(f"GRAPH:  {gr['answer']}")
    print(f"  seeds={gr['seeds']} triples={len(gr['triples'])} docs={gr['documents']}")
    return gr

_ = compare("q07")

**q04** asks which projects are affected by decisions Fatima Al-Sayed proposed. The answer needs a 2-hop walk: person → `PROPOSED` → decision → `AFFECTS` → project.

In [ ]:
_ = compare("q04")

## 4. The subgraph GraphRAG retrieves (q09: Echo's tools via dependencies)
This runs offline: it uses a hand-written query plan in place of the LLM planner.

In [ ]:
plan = llm.QueryPlan(entities=["Echo"], relation_types=["USES_TOOL", "DEPENDS_ON"], target_kinds=["tool"])
ret = graph_rag.retrieve(questions["q09"].question, plan)
print(triples_to_text(g, [t for t in ret["triples"] if t[1] in ("USES_TOOL", "DEPENDS_ON")]))
sub = g.edge_subgraph([(u, v, t) for u, t, v, _ in ret["triples"] if t in ("USES_TOOL", "DEPENDS_ON")])
draw(sub, highlight=ret["seeds"], title="q09 subgraph: Echo → depends on → uses tool", seed=3)

In [ ]:
_ = compare("q09")

## 5. Where vector RAG holds its own: passage detail
Answers that live inside prose, like numbers and rationales, are exactly what embeddings retrieve well. GraphRAG only gets them through the text of linked documents.

In [ ]:
_ = compare("q02")
_ = compare("q05")

## 6. Full evaluation
This runs both pipelines on all 12 questions and has an LLM judge grade each answer from 0 to 2 against the reference. It writes `results/eval_results.csv` and `results/comparison.md`.

In [ ]:
from genacademy_p2_rag_pipeline.evaluate import run, summary
df = run()
summary(df)

In [ ]:
scores = df.pivot_table(index="type", columns="pipeline", values="score", aggfunc="mean")[["vector", "graph"]]
ax = scores.plot.bar(figsize=(9, 4), color=["#8a8f98", "#2a78d6"], rot=0, ylim=(0, 2.1))
ax.set_ylabel("mean judge score (0-2)"); ax.set_xlabel(""); ax.set_title("Answer quality by question type")
ax.spines[["top", "right"]].set_visible(False)

In [ ]:
df.pivot(index=["id", "type"], columns="pipeline", values="score")[["vector", "graph"]]

## 7. Takeaways

- **Structure wins** when the answer is spread across relationships rather than written in one place:
  - aggregations ("who has skill A *and* B", "which projects are affected by X's decisions");
  - multi-hop chains (dependency → owner → tool, manager → report → project);
  - facts that no document states, such as who approved a decision.
- **Semantics wins** when the answer is a detail inside prose, such as a metric, a rationale or an incident narrative. The graph stores *that* a decision exists, not *why*.
- **GraphRAG's failure modes** are different:
  - If entity linking misses the right seed, nothing relevant is retrieved.
  - The graph only knows what someone modeled.
  - Hop expansion can flood the context unless edges are filtered by type.
- A practical system combines the two. It uses the graph for entities and relationships and pulls the text of linked documents for detail, which is what `GraphRAG` here already does.